# Imagen y texto en el mismo espacio

**Sesión 8 · IA Intermedio · EstudIA 2026**

En la sesión 6 convertimos frases en vectores y fotos en vectores, pero cada una en su propio espacio. Hoy usamos un
modelo que pone **fotos y frases en el mismo espacio**: una foto de un perezoso queda cerca de la frase “un perezoso
colgado de un árbol”. Con eso clasificamos fotos sin ejemplos, buscamos fotos con palabras, encontramos momentos en un
video y agrupamos fotos sin etiquetas.

El modelo es [SigLIP 2](https://huggingface.co/@@MODELO@@) de Google (licencia Apache 2.0), de la familia de CLIP, y
entiende español.

**El recorrido.**

| Parte | Qué hacemos |
|---|---|
| 1 | Una foto, varias frases |
| 2 | Clasificar sin ejemplos lo que ya conocemos |
| 3 | Buscar fotos con palabras |
| 4 | Encontrar momentos en un video |
| 5 | Agrupar fotos sin etiquetas |

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU; si tienes una, todo va más rápido.
Ejecuta la celda de preparación (▶ o `Shift + Enter`).

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, re, time, json, zipfile, urllib.request, warnings
warnings.filterwarnings("ignore")
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s08")
ARCHIVOS_S04 = ARCHIVOS.replace("/s08", "/s04")        # las imágenes de satélite y las hojas de café de la sesión 4
MODELO = "google/siglip2-base-patch16-224"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from PIL import Image
torch.set_num_threads(max(1, os.cpu_count() or 1))

In [ ]:
# @title Funciones auxiliares: descargar, escribir números, vectores de fotos y de textos, mostrar fotos
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

from transformers import AutoModel, AutoProcessor
from transformers.utils import logging as registro_hf
registro_hf.set_verbosity_error()
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
_cargados = {}

def _modelo(nombre):
    if nombre not in _cargados:
        _cargados[nombre] = (AutoModel.from_pretrained(nombre).to(DISPOSITIVO).eval(), AutoProcessor.from_pretrained(nombre))
    return _cargados[nombre]

def _unitario(v):
    v = getattr(v, "pooler_output", v)           # según la versión, viene suelto o dentro de un objeto
    return torch.nn.functional.normalize(v, dim=-1).float().cpu().numpy()

def vectores_de_fotos(fotos, modelo=None, lote=32):
    """Un vector de largo 1 por foto (fotos de PIL o rutas a archivos)."""
    red, proc = _modelo(modelo or MODELO)
    salida = []
    with torch.no_grad():
        for i in range(0, len(fotos), lote):
            grupo = [f if isinstance(f, Image.Image) else Image.open(f) for f in fotos[i:i + lote]]
            grupo = [f.convert("RGB") for f in grupo]
            salida.append(_unitario(red.get_image_features(**proc(images=grupo, return_tensors="pt").to(DISPOSITIVO))))
    return np.concatenate(salida)

def vectores_de_textos(textos, modelo=None):
    """Un vector de largo 1 por texto, en el mismo espacio que las fotos."""
    nombre = modelo or MODELO
    red, proc = _modelo(nombre)
    relleno = {"padding": "max_length", "max_length": 64} if "siglip" in nombre else {"padding": True}
    with torch.no_grad():
        entrada = proc(text=list(textos), return_tensors="pt", truncation=True, **relleno).to(DISPOSITIVO)
        return _unitario(red.get_text_features(**entrada))

def mostrar(fotos, titulos=None, columnas=6, tamano=2.0, colores=None):
    """Una cuadrícula de fotos con su título."""
    filas = (len(fotos) + columnas - 1) // columnas
    fig, ejes = plt.subplots(filas, columnas, figsize=(tamano * columnas, tamano * filas + 0.3 * filas))
    for k, eje in enumerate(np.atleast_1d(ejes).ravel()):
        eje.axis("off")
        if k < len(fotos):
            f = fotos[k] if isinstance(fotos[k], Image.Image) else Image.open(fotos[k])
            eje.imshow(f.convert("RGB"))
            if titulos:
                eje.set_title(titulos[k], fontsize=9, color=(colores[k] if colores else "black"))
    plt.tight_layout(); plt.show()

def abrir_zip(nombre):
    """Descarga (la primera vez) y descomprime un zip de la carpeta del curso; devuelve sus fotos ordenadas."""
    ruta = descargar(nombre)
    with zipfile.ZipFile(ruta) as z:
        z.extractall("archivos_curso")
        return sorted(os.path.join("archivos_curso", n) for n in z.namelist() if n.lower().endswith(".jpg"))

In [ ]:
_modelo(MODELO)
print("Listo.", "Con GPU." if DISPOSITIVO == "cuda" else "Sin GPU: funciona igual, un poco más despacio.")

---
## 1. Una foto, varias frases

El modelo tiene dos partes: una convierte fotos en vectores y otra convierte textos en vectores. Las dos se
entrenaron juntas, con cientos de millones de fotos y sus pies de foto, para que cada foto quede cerca de su texto.

Tomamos una foto de la colección de la demostración (fauna y flora de Colombia, de iNaturalist) y la comparamos con
varias frases. El número es la **similitud del coseno** de la sesión 6: mientras más grande, más cerca.

In [ ]:
fauna = abrir_zip("fauna_demo.zip")
grupos = pd.read_csv(descargar("fauna_demo.csv")).set_index("archivo").grupo
grupo_de = [grupos[os.path.relpath(f, "archivos_curso").replace(os.sep, "/")] for f in fauna]
foto = fauna[grupo_de.index("perezoso")]
FRASES = ["un perezoso colgado de un árbol", "un mono en un árbol", "una iguana sobre una rama",
          "un perro dormido", "una flor amarilla", "a sloth hanging from a tree"]
inicio = time.time()
v_foto = vectores_de_fotos([foto])[0]
v_frases = vectores_de_textos(FRASES)
print(f"Una foto y {len(FRASES)} frases en vectores en {time.time() - inicio:.1f} s.".replace(".", ",", 1))
mostrar([foto], ["La foto"], columnas=1, tamano=3)
for frase, s in sorted(zip(FRASES, v_frases @ v_foto), key=lambda x: -x[1]):
    print(f"   {s:+.3f}  {frase}".replace(".", ","))

**Para pensar.**

- ¿Qué frase quedó más cerca? ¿Y la frase en inglés?
- Los números son pequeños. ¿Importa su tamaño, o el orden?

---
## 2. Clasificar sin ejemplos lo que ya conocemos

Zero-shot, como en la sesión 6: escribimos una frase por clase y a cada foto le damos la clase cuya frase queda más
cerca. Lo probamos con dos conjuntos de la sesión 4, donde ya sabemos cuánto logró el ajuste fino:

- Las **imágenes de satélite** (10 usos del suelo, imágenes de 32 × 32 píxeles; tomamos 50 de cada clase).
- Las **hojas de café** (sana, ácaro rojo, roya leve, roya avanzada; 320 fotos).

Y probamos las frases en español y en inglés.

In [ ]:
SATELITE_ES = {'autopista': 'una autopista', 'bosque': 'un bosque', 'cultivo_anual': 'un cultivo', 'cultivo_permanente': 'un huerto de árboles frutales', 'industrial': 'una zona industrial', 'mar_o_lago': 'el mar o un lago', 'pastizal': 'un potrero', 'residencial': 'un barrio de casas', 'rio': 'un río', 'vegetacion_herbacea': 'pastos y matorrales'}
SATELITE_EN = {'autopista': 'a highway', 'bosque': 'a forest', 'cultivo_anual': 'farmland', 'cultivo_permanente': 'an orchard', 'industrial': 'an industrial area', 'mar_o_lago': 'the sea or a lake', 'pastizal': 'a pasture', 'residencial': 'a residential area', 'rio': 'a river', 'vegetacion_herbacea': 'herbaceous vegetation'}
sat = np.load(descargar("satelite_prueba.npz", f"{ARCHIVOS_S04}/satelite_prueba.npz"))
CLASES_SAT = sorted(SATELITE_ES)
elegidas = np.concatenate([np.where(sat["etiquetas"] == c)[0][:50] for c in CLASES_SAT])   # 50 por clase
sat = {"imagenes": sat["imagenes"][elegidas], "etiquetas": sat["etiquetas"][elegidas]}
inicio = time.time()
X_sat = vectores_de_fotos([Image.fromarray(a) for a in sat["imagenes"]])
print(f"{miles(len(X_sat))} imágenes de satélite en vectores en {time.time() - inicio:.0f} s.")

def zero_shot(X, clases, frases):
    """A cada foto le da la clase cuya frase queda más cerca."""
    T = vectores_de_textos([frases[c] for c in clases])
    return np.array(clases)[(X @ T.T).argmax(1)]

EXACTITUD_SAT = {}
for idioma, plantilla, frases in (("español", "una foto satelital de {}", SATELITE_ES),
                                  ("inglés", "a satellite photo of {}", SATELITE_EN)):
    pred = zero_shot(X_sat, CLASES_SAT, {c: plantilla.format(frases[c]) for c in CLASES_SAT})
    EXACTITUD_SAT[idioma] = (pred == sat["etiquetas"]).mean()
    print(f"Satélite, frases en {idioma}: {pct(EXACTITUD_SAT[idioma])}  (responder siempre lo mismo: 10 %)")
print("En la sesión 4, el modelo ajustado llegó a 96,4 % con las 2.000 imágenes de este conjunto.")
mostrar([Image.fromarray(a).resize((96, 96)) for a in sat["imagenes"][::50][:6]], list(sat["etiquetas"][::50][:6]),
        tamano=1.6)

In [ ]:
CAFE = {'sano': 'una hoja de café sana', 'acaro_rojo': 'una hoja de café con ácaro rojo', 'roya_leve': 'una hoja de café con pocas manchas de roya', 'roya_avanzada': 'una hoja de café con mucha roya anaranjada'}
ruta_cafe = descargar("cafe_entrenamiento.zip", f"{ARCHIVOS_S04}/cafe_entrenamiento.zip")
with zipfile.ZipFile(ruta_cafe) as z:
    z.extractall("archivos_curso")
    fotos_cafe = sorted(os.path.join("archivos_curso", n) for n in z.namelist() if n.endswith(".jpg"))
y_cafe = np.array([f.replace(os.sep, "/").split("/")[-2] for f in fotos_cafe])
X_cafe = vectores_de_fotos(fotos_cafe)
pred_cafe = zero_shot(X_cafe, sorted(CAFE), {c: f"una foto de {CAFE[c]}" for c in CAFE})
EXACTITUD_CAFE = (pred_cafe == y_cafe).mean()
print(f"Hojas de café, zero-shot: {pct(EXACTITUD_CAFE)}  (80 fotos por clase; responder siempre lo mismo: 25 %)")
print("En la sesión 4, el ajuste fino de todo el modelo llegó a 71,5 % de exactitud balanceada en la competencia del café.")

**Para pensar.**

- ¿Por qué crees que el zero-shot falla tanto con las imágenes de satélite y las hojas de café?
- ¿Cuántas fotos de hojas de café con roya, con su pie de foto, crees que vio el modelo en internet?

---
## 3. Buscar fotos con palabras

Las 240 fotos de la colección quedan en vectores una sola vez. Para buscar, convertimos una frase en vector y traemos
las fotos más cercanas. Es la búsqueda de la sesión 6, ahora de palabras a fotos.

In [ ]:
inicio = time.time()
X_fauna = vectores_de_fotos(fauna)
print(f"{len(fauna)} fotos en vectores en {time.time() - inicio:.0f} s.")

def buscar(frase, cuantas=4):
    s = X_fauna @ vectores_de_textos([frase])[0]
    mejores = np.argsort(-s)[:cuantas]
    print(f"“{frase}”")
    mostrar([fauna[i] for i in mejores], [f"{s[i]:.3f}".replace(".", ",") for i in mejores], columnas=cuantas, tamano=2.2)

for frase in ['un animal colgado de un árbol', 'algo amarillo con negro', 'una planta de páramo con hojas peludas', 'un animal que vive en el agua']:
    buscar(frase)

**Para pensar.**

- ¿Alguna búsqueda trajo fotos que no tienen nada que ver? ¿Qué pasa cuando lo que buscas no está en la colección?
- ¿Qué buscarías tú, con palabras, entre las fotos de tu proyecto?

---
## 4. Encontrar momentos en un video

Un video es una serie de fotos. Tomamos **una foto por segundo**, la convertimos en vector y la comparamos con una
frase. La curva de similitud en el tiempo muestra en qué momentos aparece lo que buscamos. El video junta siete clips
de Wikimedia Commons, casi todos de Colombia (los créditos están al final).

In [ ]:
import cv2
video = descargar("video_colombia.mp4")
captura = cv2.VideoCapture(video)
fps = captura.get(cv2.CAP_PROP_FPS)
cuadros, k = [], 0
while True:
    ok, cuadro = captura.read()
    if not ok:
        break
    if k % round(fps) == 0:                        # una foto por segundo
        cuadros.append(Image.fromarray(cv2.cvtColor(cuadro, cv2.COLOR_BGR2RGB)))
    k += 1
inicio = time.time()
X_video = vectores_de_fotos(cuadros)
print(f"{len(cuadros)} segundos de video, en vectores en {time.time() - inicio:.0f} s.")
mostrar(cuadros[::8], [f"segundo {s}" for s in range(0, len(cuadros), 8)], columnas=5, tamano=2.2)

In [ ]:
UMBRAL = 0.10      # por debajo, decimos que no está

def tramos(s, umbral, minimo=2):
    """Los tramos de al menos `minimo` segundos seguidos con similitud mayor o igual al umbral."""
    salida, inicio = [], None
    for t, arriba in enumerate(list(s >= umbral) + [False]):
        if arriba and inicio is None:
            inicio = t
        elif not arriba and inicio is not None:
            if t - inicio >= minimo:
                salida.append((inicio, t - 1))
            inicio = None
    return salida

FRASES_VIDEO = ['una ciudad vista desde el aire', 'un bus articulado en una avenida', 'una biblioteca en un parque', 'un colibrí', 'una persona en bicicleta']
fig, eje = plt.subplots(figsize=(10, 3.4))
colores = ["#FF7C1A", "#CB318E", "#7FC200", "#000000", "#808080"]
for frase, color in zip(FRASES_VIDEO, colores):
    s = X_video @ vectores_de_textos([frase])[0]
    eje.plot(range(len(s)), s, color=color, lw=2, label=frase)
    t = tramos(s, UMBRAL)
    print(f"{frase:32s} → " + (", ".join(f"del segundo {a} al {b}" for a, b in t) + f" (lo más parecido: el {s.argmax()})"
                                 if t else "no aparece"))
eje.axhline(UMBRAL, color="#B0B0B0", ls="--", lw=1)
eje.set_xlabel("Segundo del video"); eje.set_ylabel("Similitud con la frase")
eje.legend(frameon=False, fontsize=8, ncol=2, loc="upper right"); eje.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**Para pensar.**

- ¿En qué segundos aparece el colibrí? ¿Qué pasa con la persona en bicicleta?
- Si tuvieras horas de video de una cámara, ¿qué buscarías? ¿Qué tan importante es elegir bien el umbral?

---
## 5. Agrupar fotos sin etiquetas

Sin decirle al modelo qué hay en las fotos, agrupamos los vectores con **k-medias** (*k-means*): busca 8 grupos de fotos
cercanas entre sí. Después le ponemos nombre a cada grupo: comparamos el centro del grupo con una lista de palabras.

In [ ]:
from sklearn.cluster import KMeans
NOMBRES = ['una ardilla', 'un frailejón', 'una iguana', 'una mariposa monarca', 'un mono aullador', 'un perezoso', 'una rana venenosa', 'una abeja', 'un pájaro', 'un perro', 'una flor', 'un árbol', 'una serpiente', 'un gato', 'una tortuga', 'una araña']
T_nombres = vectores_de_textos([f"una foto de {n}" for n in NOMBRES])
kmedias = KMeans(n_clusters=8, n_init=10, random_state=0).fit(X_fauna)
for g in range(8):
    centro = kmedias.cluster_centers_[g] / np.linalg.norm(kmedias.cluster_centers_[g])
    nombre = NOMBRES[int(np.argmax(T_nombres @ centro))]
    miembros = np.where(kmedias.labels_ == g)[0]
    print(f"Grupo {g + 1}: {len(miembros)} fotos · nombre más cercano: {nombre}")
    mostrar([fauna[i] for i in miembros[:6]], columnas=6, tamano=1.5)

In [ ]:
# Las fotos sí traen su grupo real (no lo usamos para agrupar). ¿Qué tan bien quedaron?
tabla = pd.crosstab(pd.Series(kmedias.labels_ + 1, name="grupo encontrado"), pd.Series(grupo_de, name="grupo real"))
display(tabla)
PUREZA = tabla.max(axis=1).sum() / len(fauna)
print(f"Pureza: {pct(PUREZA)} de las fotos quedaron en un grupo donde su grupo real es el más común.")

**Para pensar.**

- ¿Qué grupo quedó mezclado? ¿Qué tienen en común esas fotos?
- ¿Para qué te serviría agrupar sin etiquetas antes de etiquetar los datos de un proyecto?

**Lo que viene.** En la competencia de hoy clasificas aves de Colombia solo con palabras. En la sesión 9 usamos la
búsqueda por significado para darle información a un modelo de lenguaje: un sistema RAG.

**Fuentes.** Modelo: Tschannen, M. y otros (2025). SigLIP 2. `google/siglip2-base-patch16-224`, Apache 2.0. Fotos de
fauna y flora: observaciones de iNaturalist hechas en Colombia, con licencia CC0, CC BY o CC BY-SA; el autor y la
licencia de cada foto están en `creditos_fotos.csv`. Video: clips de Wikimedia Commons de Neoredacturus (Pereira, CC
BY-SA 4.0), Managementboy (TransMilenio, CC BY-SA 3.0), Yeison Medina Medina (Quebrada Santa Elena, CC BY-SA 4.0),
ProtoplasmaKid (Plaza de Bolívar, CC BY-SA 4.0), MrMatz07 (Parque El Tunal, CC BY-SA 4.0), Anni Roskilde (colibríes en
Costa Rica, CC BY 3.0) y 4mduc14s99 (Bogotá, CC BY-SA 4.0); el video armado se comparte con CC BY-SA 4.0. Imágenes de
satélite y hojas de café: las de la sesión 4 (EuroSAT, MIT; RoCoLe, CC BY 4.0).